In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print('Hello World')


In [ ]:
def load_waiting_times(path=None, df=None):
    """ This function loads the waiting times from the simulation results
    path: str - the path to the simulation results
    df: pd.DataFrame - the simulation results
    """
    if df is None:
        if path is None:
            raise ValueError("Provide either path or df")
        df = pd.read_csv(path, skipinitialspace=True)

    df = df.copy()
    df.columns = df.columns.str.strip()

    datetime_cols = ['Screening Queue', 'Screening Start', 'Reporting Queue', 'Reporting Start']
    for col in datetime_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .replace('N/A', pd.NA)
        )
        df[col] = pd.to_datetime(df[col], errors='coerce')

    df['waiting_for_screening_days'] = (
        df['Screening Start'] - df['Screening Queue']
    ).dt.total_seconds() / (24 * 3600)

    df['waiting_for_reporting_days'] = (
        df['Reporting Start'] - df['Reporting Queue']
    ).dt.total_seconds() / (24 * 3600)

    return df

dfs = {run_id: load_waiting_times(path) for run_id, path in RUNS.items()}

for run_id, df in dfs.items():
    print(f"\n{run_id} run: {len(df)} patients")
    display(df[['Patient ID', 'waiting_for_screening_days', 'waiting_for_reporting_days']].head())

In [ ]:
for run_id, df in dfs.items():
    fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

    axes[0].plot(df.index, df['waiting_for_screening_days'], linewidth=1, alpha=0.7, color='steelblue')
    axes[0].set_ylabel('Waiting Time (days)')
    axes[0].set_title(f'Waiting Time for Screening ({run_id} run)', fontweight='bold')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(df.index, df['waiting_for_reporting_days'], linewidth=1, alpha=0.7, color='purple')
    axes[1].set_xlabel('Patient Number')
    axes[1].set_ylabel('Waiting Time (days)')
    axes[1].set_title(f'Waiting Time for Reporting ({run_id} run)', fontweight='bold')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    for label, col in [('Screening', 'waiting_for_screening_days'), ('Reporting', 'waiting_for_reporting_days')]:
        print(f"\n{run_id} - Waiting Time for {label} Statistics:")
        print(f"  Mean:   {df[col].mean():.3f} days")
        print(f"  Median: {df[col].median():.3f} days")
        print(f"  Std:    {df[col].std():.3f} days")
        print(f"  Min:    {df[col].min():.3f} days")
        print(f"  Max:    {df[col].max():.3f} days")

In [ ]:
for run_id, df in dfs.items():
    plt.figure(figsize=(16, 6))
    plt.plot(df.index, df['waiting_for_screening_days'], label='Waiting for Screening',
             linewidth=1.5, alpha=0.7, color='steelblue')
    plt.plot(df.index, df['waiting_for_reporting_days'], label='Waiting for Reporting',
             linewidth=1.5, alpha=0.7, color='purple')
    plt.xlabel('Patient Number')
    plt.ylabel('Waiting Time (days)')
    plt.title(f'Screening and Reporting Waiting Times ({run_id} run)', fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
SIM_RESULTS = '../sim_results/20260705_174955'

res_util_pct = pd.read_csv(f'{SIM_RESULTS}_resource_busy_intervals.csv', skipinitialspace=True)
res_util_pct.columns = res_util_pct.columns.str.strip()
res_util_pct['utilisation_key'] = res_util_pct['utilisation_key'].str.strip()

metadata = pd.read_csv(f'{SIM_RESULTS}_resource_metadata.csv', skipinitialspace=True)
metadata.columns = metadata.columns.str.strip()
metadata['utilisation_key'] = metadata['utilisation_key'].str.strip()

busy_by_key = (
    res_util_pct
    .groupby('utilisation_key', as_index=False)['duration_min']
    .sum()
    .rename(columns={'duration_min': 'busy_server_minutes'})
)

cytotech_scheduled_minutes = metadata.loc[
    metadata['utilisation_key'] == 'Cytotechnicians',
    'scheduled_available_minutes',
].iloc[0]

utilisation_rows = []
for _, meta_row in metadata.iterrows():
    key = meta_row['utilisation_key']
    busy_minutes = float(
        busy_by_key.loc[busy_by_key['utilisation_key'] == key, 'busy_server_minutes'].iloc[0]
    )
    capacity = float(meta_row['capacity'])
    scheduled_available_minutes = (
        cytotech_scheduled_minutes
        if key == 'Manual Staining Station'
        else float(meta_row['scheduled_available_minutes'])
    )
    capacity_minutes = capacity * scheduled_available_minutes
    utilisation_pct = (busy_minutes / (capacity_minutes) * 100) if capacity_minutes > 0 else 0.0

    utilisation_rows.append({
        'utilisation_key': key,
        'busy_server_minutes': busy_minutes,
        'scheduled_available_minutes': scheduled_available_minutes,
        'capacity': capacity,
        'capacity_minutes': capacity_minutes,
        'utilisation_pct': round(utilisation_pct, 2),
    })

resource_utilisation = pd.DataFrame(utilisation_rows).sort_values('utilisation_key').reset_index(drop=True)
resource_utilisation



In [ ]:
plt.figure(figsize=(16, 6))
bars = plt.bar(
    resource_utilisation['utilisation_key'],
    resource_utilisation['utilisation_pct'],
    color='steelblue',
)
plt.bar_label(
    bars,
    labels=[f'{v:.1f}%' for v in resource_utilisation['utilisation_pct']],
    padding=3,
    fontsize=12,
)
plt.ylim(0, 100)
plt.ylabel('Utilisation (%)', fontsize=14)
plt.xlabel('Resource', fontsize=14)
plt.title('Resource Utilisation', fontweight='bold')
plt.xticks(ha='center', fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


In [ ]:
SIM_RESULTS = '../sim_results/20260706_112628' # 3 Junior pathologists

res_util_pct = pd.read_csv(f'{SIM_RESULTS}_resource_busy_intervals.csv', skipinitialspace=True)
res_util_pct.columns = res_util_pct.columns.str.strip()
res_util_pct['utilisation_key'] = res_util_pct['utilisation_key'].str.strip()

metadata = pd.read_csv(f'{SIM_RESULTS}_resource_metadata.csv', skipinitialspace=True)
metadata.columns = metadata.columns.str.strip()
metadata['utilisation_key'] = metadata['utilisation_key'].str.strip()

busy_by_key = (
    res_util_pct
    .groupby('utilisation_key', as_index=False)['duration_min']
    .sum()
    .rename(columns={'duration_min': 'busy_server_minutes'})
)

cytotech_scheduled_minutes = metadata.loc[
    metadata['utilisation_key'] == 'Cytotechnicians',
    'scheduled_available_minutes',
].iloc[0]

utilisation_rows = []
for _, meta_row in metadata.iterrows():
    key = meta_row['utilisation_key']
    busy_minutes = float(
        busy_by_key.loc[busy_by_key['utilisation_key'] == key, 'busy_server_minutes'].iloc[0]
    )
    capacity = float(meta_row['capacity'])
    scheduled_available_minutes = (
        cytotech_scheduled_minutes
        if key == 'Manual Staining Station'
        else float(meta_row['scheduled_available_minutes'])
    )
    capacity_minutes = capacity * scheduled_available_minutes
    utilisation_pct = (busy_minutes / (capacity_minutes) * 100) if capacity_minutes > 0 else 0.0

    utilisation_rows.append({
        'utilisation_key': key,
        'busy_server_minutes': busy_minutes,
        'scheduled_available_minutes': scheduled_available_minutes,
        'capacity': capacity,
        'capacity_minutes': capacity_minutes,
        'utilisation_pct': round(utilisation_pct, 2),
    })

resource_utilisation = pd.DataFrame(utilisation_rows).sort_values('utilisation_key').reset_index(drop=True)
resource_utilisation



In [ ]:
plt.figure(figsize=(16, 6))
colors = [
    'indianred' if pct > 100 else 'steelblue'
    for pct in resource_utilisation['utilisation_pct']
]
bars = plt.bar(
    resource_utilisation['utilisation_key'],
    resource_utilisation['utilisation_pct'],
    color=colors,
)
plt.bar_label(
    bars,
    labels=[f'{v:.1f}%' for v in resource_utilisation['utilisation_pct']],
    padding=3,
    fontsize=12,
)
plt.axhline(100, color='black', linestyle='--', linewidth=1, alpha=0.6, label='100% capacity')
plt.ylim(0, max(100, resource_utilisation['utilisation_pct'].max() + 10))
plt.ylabel('Utilisation (%)', fontsize=14)
plt.xlabel('Resource', fontsize=14)
plt.title('Resource Utilisation — red capacity run (3 junior pathologists)', fontweight='bold')
plt.xticks(rotation=30, ha='right', fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
red_capacity = pd.read_csv(r'/Users/varad/Desktop/Lenovo Desktop/Oxford/DPhil Cervical Cancer Scale-up/Code/real_world_sim/sim_results/20260706_112628_simulation_patient_timestamps.csv')
red_capacity.head()

red_capacity = load_waiting_times('../sim_results/20260706_112628_simulation_patient_timestamps.csv')
red_capacity['Type'] = red_capacity['Type'].str.strip()
red_capacity.head()

In [ ]:
pap_smear = red_capacity[red_capacity['Type'] == 'pap_smear'].reset_index(drop=True)
non_pap_smear = red_capacity[red_capacity['Type'] == 'non_pap_smear'].reset_index(drop=True)

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=False)
panels = [
    (pap_smear, 'Pap smear', axes[0]),
    (non_pap_smear, 'Non-pap smear', axes[1]),
]

for df, label, ax in panels:
    patient_numbers = np.arange(1, len(df) + 1)
    ax.plot(
        patient_numbers,
        df['waiting_for_screening_days'],
        label='Waiting for screening',
        linewidth=1.2,
        alpha=0.8,
        color='steelblue',
    )
    ax.plot(
        patient_numbers,
        df['waiting_for_reporting_days'],
        label='Waiting for reporting',
        linewidth=1.2,
        alpha=0.8,
        color='purple',
    )
    ax.set_title(f'{label} patients (n={len(df):,})', fontweight='bold', fontsize=14)
    ax.set_ylabel('Waiting time (days)', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=11)

axes[1].set_xlabel('Patient number', fontsize=12)
fig.suptitle('Red capacity run: waiting times by patient type', fontweight='bold', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()